In [38]:
import requests
import json
OVERPASS_URL = "https://overpass.kumi.systems/api/interpreter"

In [39]:
HEADERS = {
    "User-Agent": "DataCenterGlobalMappingProject/2.1 (Academic Research Script)"
}

In [41]:
import requests
import pandas as pd
import time

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# FIX: Completely custom User-Agent with NO placeholder text/emails to bypass the 406 block
HEADERS = {
    "User-Agent": "DataCenterGlobalMappingProject/2.1 (Academic Research Script)"
}

# Split the world into 4 manageable coordinate boxes (South, West, North, East)
QUADRANTS = {
    "North-West (Americas)": (0, -180, 90, 0),
    "North-East (Europe/Asia)": (0, 0, 90, 180),
    "South-West (Latin America)": (-90, -180, 0, 0),
    "South-East (Africa/Oceania)": (-90, 0, 0, 180)
}

all_data_centers = []

print("Starting quadrant-based global search...")

for name, bbox in QUADRANTS.items():
    print(f"Fetching data for {name} quadrant...")
    s, w, n, e = bbox
    
    # Using raw bounding boxes is incredibly fast compared to area lookups
    query = f"""
    [out:json][timeout:60];
    (
      nwr["telecom"="data_center"]({s},{w},{n},{e});
      nwr["building"="data_center"]({s},{w},{n},{e});
    );
    out center;
    """
    
    try:
        # 60s is more than enough for a bounding box
        response = requests.post(OVERPASS_URL, data={'data': query}, headers=HEADERS, timeout=60)
        response.raise_for_status()
        raw_data = response.json()
        
        elements = raw_data.get('elements', [])
        print(f"-> Found {len(elements)} data centers.")
        
        for item in elements:
            tags = item.get('tags', {})
            lat = item.get('lat') or item.get('center', {}).get('lat')
            lon = item.get('lon') or item.get('center', {}).get('lon')
            
            all_data_centers.append({
                "Name": tags.get("name", "Unnamed Facility"),
                "Operator": tags.get("operator", "Unknown Operator"),
                "Quadrant": name,
                "Country Code": tags.get("addr:country", "N/A"),
                "City": tags.get("addr:city", "N/A"),
                "Latitude": lat,
                "Longitude": lon,
                "OSM ID": item.get("id")
            })
            
        # Give the server a 3-second break between quadrants
        time.sleep(3)
        
    except Exception as err:
        print(f"❌ Failed to fetch data for {name}: {err}")

# Build and display the clean DataFrame
df_global = pd.DataFrame(all_data_centers)

# Drop any duplicate IDs just in case items fall right on the boundaries
df_global.drop_duplicates(subset=["OSM ID"], inplace=True)

print(f"\nFinished! Combined Global Dataset contains {len(df_global)} unique data centers.")
df_global.head(15)

Starting quadrant-based global search...
Fetching data for North-West (Americas) quadrant...
-> Found 2276 data centers.
Fetching data for North-East (Europe/Asia) quadrant...
-> Found 1885 data centers.
Fetching data for South-West (Latin America) quadrant...
-> Found 143 data centers.
Fetching data for South-East (Africa/Oceania) quadrant...
-> Found 269 data centers.

Finished! Combined Global Dataset contains 4573 unique data centers.


,Name,Operator,Quadrant,Country Code,City,Latitude,Longitude,OSM ID
0,CoreSite DC1,CoreSite,North-West (Americas),N/A,N/A,38.902888,-77.029149,751881301
1,Southwest Cyberport,Unknown Operator,North-West (Americas),N/A,Albuquerque,35.103293,-106.589349,1825907839
2,SDSU Computer Room,Unknown Operator,North-West (Americas),N/A,N/A,32.775025,-117.070888,2607827436
3,Equinix TR1,Equinix,North-West (Americas),N/A,Toronto,43.644529,-79.384389,2733736931
4,NocRoom Miami IT Services,Unknown Operator,North-West (Americas),N/A,N/A,25.791545,-80.317748,3406467087
5,NocRoom Miami IT Services,Unknown Operator,North-West (Americas),N/A,N/A,25.646562,-80.423435,3406476814
6,Center for Computation & Technology (CCT),Unknown Operator,North-West (Americas),N/A,N/A,30.407446,-91.172545,3468203102
7,The Pittock Internet Exchange,Alco Properties,North-West (Americas),N/A,Portland,45.521425,-122.680562,3732415309
8,Google Fiber,Google,North-West (Americas),N/A,N/A,33.771568,-84.366859,4879481820
9,Pathway Communications,Unknown Operator,North-West (Americas),N/A,Markham,43.857356,-79.356543,4922135764


In [42]:
df_global

,Name,Operator,Quadrant,Country Code,City,Latitude,Longitude,OSM ID
0,CoreSite DC1,CoreSite,North-West (Americas),N/A,N/A,38.902888,-77.029149,751881301
1,Southwest Cyberport,Unknown Operator,North-West (Americas),N/A,Albuquerque,35.103293,-106.589349,1825907839
2,SDSU Computer Room,Unknown Operator,North-West (Americas),N/A,N/A,32.775025,-117.070888,2607827436
3,Equinix TR1,Equinix,North-West (Americas),N/A,Toronto,43.644529,-79.384389,2733736931
4,NocRoom Miami IT Services,Unknown Operator,North-West (Americas),N/A,N/A,25.791545,-80.317748,3406467087
...,...,...,...,...,...,...,...,...
4568,Digital Hyperspace Jakarta,Digital Hyperspace Indonesia,South-East (Africa/Oceania),N/A,N/A,-6.370178,107.191972,1533569835
4569,STT Jakarta 1,Singapore Technologies Telemedia,South-East (Africa/Oceania),N/A,N/A,-6.374010,107.201067,1533569836
4570,Amazon Web Services,Amazon Web Services,South-East (Africa/Oceania),N/A,N/A,-6.420064,107.355543,1533569837
4571,DigiCo BNE1,DigiCo,South-East (Africa/Oceania),N/A,N/A,-27.415833,153.091456,19402810
